# Colab AI + GitHub Starter 🚀

This notebook calls an AI API, then uses the GitHub REST API to create a repository and publish starter files. **Keep tokens in Colab Secrets, never in notebook code.**


## 1. Install packages
Run this cell first.


In [ ]:
%pip -q install openai requests


## 2. Connect Colab Secrets
In the left sidebar, click the **key icon (Secrets)** and add `OPENAI_API_KEY` and `GITHUB_TOKEN`. Enable notebook access for each. Then run this cell.


In [ ]:
app_code = r'''import os
import requests
from openai import OpenAI

# The OpenAI API key is read from an environment variable.
# In Google Colab, set it in the Secrets panel as OPENAI_API_KEY.
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def ask_ai(question: str) -> str:
    """Send a question to the AI API and return its answer."""
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=question
    )
    return response.output_text

def publish_to_github(repo_name: str, description: str, files: dict[str, str]) -> str:
    """
    Create a public GitHub repository and upload files using a GitHub Personal
    Access Token (PAT). `files` maps file paths to their text contents.
    """
    token = os.environ["GITHUB_TOKEN"]
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    # Create the repository under the account that owns the token.
    response = requests.post(
        "https://api.github.com/user/repos",
        headers=headers,
        json={
            "name": repo_name,
            "description": description,
            "private": False,
            "auto_init": True,
        },
        timeout=30,
    )
    if response.status_code == 422:
        raise RuntimeError(
            f"Repository '{repo_name}' may already exist. Choose a new name."
        )
    response.raise_for_status()
    repo = response.json()
    owner = repo["owner"]["login"]
    repo_name = repo["name"]

    # Upload each file through GitHub's Contents API.
    for path, content in files.items():
        upload = requests.put(
            f"https://api.github.com/repos/{owner}/{repo_name}/contents/{path}",
            headers=headers,
            json={
                "message": f"Add {path}",
                "content": __import__("base64").b64encode(
                    content.encode("utf-8")
                ).decode("ascii"),
            },
            timeout=30,
        )
        upload.raise_for_status()

    return repo["html_url"]
'''exec(app_code)


## 3. Create the tiny AI app
The app accepts a question and prints the model's answer.


In [ ]:
import os
import requests
from openai import OpenAI

# The OpenAI API key is read from an environment variable.
# In Google Colab, set it in the Secrets panel as OPENAI_API_KEY.
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def ask_ai(question: str) -> str:
    """Send a question to the AI API and return its answer."""
    response = client.responses.create(
        model="gpt-4.1-mini",
        input=question
    )
    return response.output_text

def publish_to_github(repo_name: str, description: str, files: dict[str, str]) -> str:
    """
    Create a public GitHub repository and upload files using a GitHub Personal
    Access Token (PAT). `files` maps file paths to their text contents.
    """
    token = os.environ["GITHUB_TOKEN"]
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    # Create the repository under the account that owns the token.
    response = requests.post(
        "https://api.github.com/user/repos",
        headers=headers,
        json={
            "name": repo_name,
            "description": description,
            "private": False,
            "auto_init": True,
        },
        timeout=30,
    )
    if response.status_code == 422:
        raise RuntimeError(
            f"Repository '{repo_name}' may already exist. Choose a new name."
        )
    response.raise_for_status()
    repo = response.json()
    owner = repo["owner"]["login"]
    repo_name = repo["name"]

    # Upload each file through GitHub's Contents API.
    for path, content in files.items():
        upload = requests.put(
            f"https://api.github.com/repos/{owner}/{repo_name}/contents/{path}",
            headers=headers,
            json={
                "message": f"Add {path}",
                "content": __import__("base64").b64encode(
                    content.encode("utf-8")
                ).decode("ascii"),
            },
            timeout=30,
        )
        upload.raise_for_status()

    return repo["html_url"]


## 4. Try the AI API
Change the sample question if you like, then run this cell.


In [ ]:
project_readme = """# Colab AI + GitHub Starter\n\nA small Python AI app run from Google Colab. It uses an AI API key for responses and the GitHub REST API to publish project files.\n\n## Run it\nOpen the notebook in Google Colab, add `OPENAI_API_KEY` and `GITHUB_TOKEN` in Colab Secrets, enable notebook access, and run cells top to bottom.\n\n## Security\nKeep tokens in Colab Secrets. Never commit or share them.\n"""with open("app.py", "w", encoding="utf-8") as f:    f.write(app_code)repo_url = publish_to_github(    repo_name="colab-ai-github-starter-CHANGE-ME",    description="Beginner AI API project run in Google Colab",    files={"app.py": app_code, "README.md": project_readme},)print("Published repository:", repo_url)


## 5. Publish the project to GitHub
Choose a repository name that is not already in your GitHub account. This creates a **public** repository and uploads `app.py` and `README.md`. Do not put secrets in either file.


In [ ]:
project_readme = """# Colab AI + GitHub Starter\n\nA small Python AI app run from Google Colab. It uses an AI API key for responses and the GitHub REST API to publish project files.\n\n## Run it\nOpen the notebook in Google Colab, add `OPENAI_API_KEY` and `GITHUB_TOKEN` in Colab Secrets, enable notebook access, and run cells top to bottom.\n\n## Security\nKeep tokens in Colab Secrets. Never commit or share them.\n"""
with open("app.py", "w", encoding="utf-8") as f:
    f.write(__import__("inspect").getsource(ask_ai) + "\n\n" + __import__("inspect").getsource(publish_to_github))

repo_url = publish_to_github(
    repo_name="colab-ai-github-starter-CHANGE-ME",
    description="Beginner AI API project run in Google Colab",
    files={"app.py": open("app.py", encoding="utf-8").read(), "README.md": project_readme},
)
print("Published repository:", repo_url)


## 6. Add this notebook to the repository
In Colab, choose **File → Download → Download .ipynb**. Open the repository URL printed above, choose **Add file → Upload files**, and upload the downloaded notebook. This keeps the runnable notebook alongside `app.py` and `README.md`.

**If you accidentally expose a token:** revoke it immediately in the provider's settings, create a replacement, and remove the exposed value from the notebook and repository history.
